In [11]:
import pandas as pd
import sqlite3

In [12]:
df=pd.read_csv("C:/Users/Admin/.cache/kagglehub/datasets/yashch05/direct-to-consumer-e-commerce-funnel-dataset/versions/1/d2c_marketing_funnel_data.csv")

In [13]:
df.head()

,user_id,session_id,date,month,channel,campaign_type,device,user_type,region,visited_website,viewed_product,added_to_cart,checkout_started,purchase_completed,discount_applied,order_value,revenue
0,221958,1,8/16/2025,2025-08,Organic,New Launch,Mobile,New,Metro,Yes,No,No,No,No,No,499.00,0.000
1,771155,2,12/16/2025,2025-12,Organic,Influencer,Mobile,New,Non-Metro,Yes,Yes,Yes,No,No,No,499.00,0.000
2,231932,3,7/17/2025,2025-07,Organic,Influencer,Mobile,New,Non-Metro,Yes,Yes,No,No,No,No,499.00,0.000
3,465838,4,7/4/2025,2025-07,Paid Ads,Discount,Mobile,Returning,Metro,Yes,Yes,Yes,Yes,Yes,Yes,2000.95,1800.855
4,359178,5,8/10/2025,2025-08,Paid Ads,Influencer,Mobile,Returning,Non-Metro,Yes,No,No,No,No,No,499.00,0.000


In [14]:
conn = sqlite3.connect("ecommerce.db")

In [15]:
df.to_sql("ecommerce_data", conn, if_exists="replace", index=False)

120000

In [17]:
pd.read_sql(
"""
SELECT *
FROM ecommerce_data
LIMIT 5
""",
conn)


,user_id,session_id,date,month,channel,campaign_type,device,user_type,region,visited_website,viewed_product,added_to_cart,checkout_started,purchase_completed,discount_applied,order_value,revenue
0,221958,1,8/16/2025,2025-08,Organic,New Launch,Mobile,New,Metro,Yes,No,No,No,No,No,499.00,0.000
1,771155,2,12/16/2025,2025-12,Organic,Influencer,Mobile,New,Non-Metro,Yes,Yes,Yes,No,No,No,499.00,0.000
2,231932,3,7/17/2025,2025-07,Organic,Influencer,Mobile,New,Non-Metro,Yes,Yes,No,No,No,No,499.00,0.000
3,465838,4,7/4/2025,2025-07,Paid Ads,Discount,Mobile,Returning,Metro,Yes,Yes,Yes,Yes,Yes,Yes,2000.95,1800.855
4,359178,5,8/10/2025,2025-08,Paid Ads,Influencer,Mobile,Returning,Non-Metro,Yes,No,No,No,No,No,499.00,0.000


In [ ]:
№1. Общие KPI

In [18]:
query = """
SELECT
COUNT(DISTINCT user_id) AS users,
COUNT(DISTINCT session_id) AS sessions,
SUM(revenue) AS revenue
FROM ecommerce_data
"""

pd.read_sql(query, conn)

,users,sessions,revenue
0,112352,120000,1.701660e+07


In [ ]:
№2. Воронка

In [19]:
query = """
SELECT
SUM(CASE WHEN visited_website='Yes' THEN 1 ELSE 0 END) AS visited,
SUM(CASE WHEN viewed_product='Yes' THEN 1 ELSE 0 END) AS viewed,
SUM(CASE WHEN added_to_cart='Yes' THEN 1 ELSE 0 END) AS cart,
SUM(CASE WHEN checkout_started='Yes' THEN 1 ELSE 0 END) AS checkout,
SUM(CASE WHEN purchase_completed='Yes' THEN 1 ELSE 0 END) AS purchase
FROM ecommerce_data
"""
pd.read_sql(query, conn)

,visited,viewed,cart,checkout,purchase
0,120000,77870,27156,16234,8181


In [ ]:
№3. Каналы

In [20]:
query = """
SELECT
channel,
COUNT(*) AS purchases,
SUM(revenue) AS revenue
FROM ecommerce_data
WHERE purchase_completed='Yes'
GROUP BY channel
ORDER BY revenue DESC
"""

pd.read_sql(query, conn)

,channel,purchases,revenue
0,Paid Ads,3619,7536147.080
1,Organic,2448,5090708.447
2,Social,1230,2578443.312
3,Email,884,1811300.315


In [ ]:
№4. Revenue

In [21]:
query = """
SELECT

channel,

COUNT(DISTINCT user_id) AS users,

SUM(revenue) AS revenue,

ROUND(
SUM(revenue) * 1.0 /
COUNT(DISTINCT user_id),
2
) AS revenue_per_user

FROM ecommerce_data

GROUP BY channel

ORDER BY revenue_per_user DESC
"""

pd.read_sql(query, conn)

,channel,users,revenue,revenue_per_user
0,Email,12019,1811300.315,150.70
1,Organic,35225,5090708.447,144.52
2,Social,17897,2578443.312,144.07
3,Paid Ads,52346,7536147.080,143.97


In [ ]:
№5. Новые и возвращающиеся пользователи

In [22]:
query = """
SELECT

user_type,

COUNT(*) AS purchases,

AVG(order_value) AS avg_order,

SUM(revenue)

AS revenue

FROM ecommerce_data

WHERE purchase_completed='Yes'

GROUP BY user_type
"""

pd.read_sql(query, conn)

,user_type,purchases,avg_order,revenue
0,New,5398,2200.401206,1.122580e+07
1,Returning,2783,2199.092429,5.790797e+06


In [ ]:
№6. Скидки

In [23]:
query = """
SELECT

discount_applied,

COUNT(*) AS purchases,

AVG(order_value) AS avg_order,

SUM(revenue) AS revenue

FROM ecommerce_data

WHERE purchase_completed='Yes'

GROUP BY discount_applied
"""

pd.read_sql(query, conn)

,discount_applied,purchases,avg_order,revenue
0,No,3719,2200.976628,8185432.080
1,Yes,4462,2199.105303,8831167.074


In [ ]:
Вывод

SQL-анализ подтвердил результаты,
полученные ранее с помощью Pandas.

Были рассчитаны:

- продуктовые KPI
- показатели воронки
- эффективность каналов
- влияние скидок
- различия между сегментами пользователей